# Atlas-100M, second GPU: opponents only (ParadoxAI Lab)

Runs every opponent that does not need Atlas-100M — nine foundation models on all-India PM2.5; TimesFM / Chronos,
STID, Graph WaveNet, STAEformer and PWM-Dyn on the six traffic networks — and downloads the global systems' forecast
archive for the hybrid stage, **while the main notebook trains on another account**. Same rules
(`atlas/m100/PREREG.md`), same Drive store (`PWM-100M/`), resumable: after a disconnect, **Run all** again.

Start the main notebook first (its data step builds the shared all-India file), then this one. In the Drive popup,
choose the company account (contact.paradoxailab@gmail.com) even if Colab runs under another account.


In [ ]:
# 1. Drive and GPU
import os, subprocess
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)
STORE = '/content/drive/MyDrive/PWM-100M'; W10 = '/content/drive/MyDrive/PWM-10M'
assert os.path.exists(f'{W10}/data/W5K_X.npy'), 'the WEATHER-5K copy (PWM-10M/data) is missing from Drive'
assert os.path.exists(f'{W10}/live/W5K_OBS.npy'), 'the NOAA live-period copy (PWM-10M/live) is missing from Drive'
os.makedirs(STORE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Code (public repository) and packages
!rm -rf /content/pwm && git clone -q --depth 1 -b atlas-v0.1 https://github.com/paradoxailab/paradox-world-model.git /content/pwm
!cd /content/pwm && git log --oneline -1
!pip install -q timesfm chronos-forecasting kagglehub 2>&1 | tail -1
!pip install -q --no-deps granite-tsfm tirex-ts && pip install -q einops deprecated 2>&1 | tail -1

In [ ]:
# 3. Data, then every opponent and the forecast archive (no Atlas-100M needed)
P = 'python -u /content/pwm/atlas/m100/pipeline.py --store ' + STORE + ' --w10 ' + W10
!{P} --stage data,opponents 2>&1 | grep --line-buffered -v "Warning\|warn(" | tail -120
